# 08 — Hadamard & Measurement Bases

Meet the gate every circuit opens with, and turn it around to *measure in a different basis* — the recipe that explains the `h` and `sdg` you will see in `01/11` and `01/15`.

**Prerequisites:** `01/07_pauli_gates`.

**What you'll be able to do**
- Apply the Hadamard gate $H$ and see it map the $Z$ basis to the $X$ basis ($H|0\rangle=|+\rangle$, $HZH=X$).
- Measure a qubit in the $X$ or $Y$ basis by *rotating that basis onto $Z$* before measuring.
- Explain why a circuit starts with `qc.h(0)`, and why $Y$-basis read-out needs an `sdg` first.

A computational-basis ($Z$) measurement is the only kind real hardware performs. Yet `01/03` showed it is blind to phase. The Hadamard is how we get around that: it rotates the interesting basis onto $Z$ so the measurement we *can* do answers the question we *want* to ask.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.quantum.states import KET_0, KET_1, KET_PLUS, KET_MINUS, qubit_state, born_probabilities
from qot_course.quantum.gates import HADAMARD, PAULI_Z, PAULI_X, S_DAG, apply_gate

np.random.seed(42)
viz.use_course_style()

## 1. The Hadamard maps the Z basis to the X basis

$$ H = \frac{1}{\sqrt2}\begin{pmatrix}1&1\\1&-1\end{pmatrix}. $$

$H$ sends the computational basis to the superposition basis: $H|0\rangle = |+\rangle$ and $H|1\rangle = |-\rangle$. It is its own inverse ($H^2 = I$), so it also sends $|+\rangle \to |0\rangle$ and $|-\rangle \to |1\rangle$. And it exchanges the $X$ and $Z$ observables: $HZH = X$.

In [ ]:
print("H|0> =", np.round(apply_gate(HADAMARD, KET_0), 3), " (= |+>)")
print("H|1> =", np.round(apply_gate(HADAMARD, KET_1), 3), " (= |->)")
print("H|+> =", np.round(apply_gate(HADAMARD, KET_PLUS), 3), " (= |0>: H undoes itself)")
print("HZH  =\n", np.round(HADAMARD @ PAULI_Z @ HADAMARD, 3), " (= X)")
print("HZH == X ?", np.allclose(HADAMARD @ PAULI_Z @ HADAMARD, PAULI_X))

**Read the output.** $H$ swaps the two bases both ways, and conjugating $Z$ by $H$ produces $X$. That last identity is the key: it means "ask the $X$ question" can be rewritten as "apply $H$, then ask the $Z$ question". This is the gate a circuit opens with (`qc.h(0)`) to leave the classical world for superposition.

## 2. Measuring in the X basis = rotate, then measure Z

Hardware only measures in $Z$ (outcomes $0/1$). To measure in the $X$ basis — to ask "is this state $|+\rangle$ or $|-\rangle$?" — we rotate the $X$ basis onto $Z$ with $H$, then do the $Z$ measurement:

$$ P_X(\pm) = P_Z\big(0/1 \text{ on } H|\psi\rangle\big). $$

Watch the difference on $|+\rangle$: blind in $Z$, certain in $X$.

In [ ]:
def probabilities_in_basis(state, basis):
    # Born probabilities of `state` measured in the X, Y, or Z basis.
    if basis == "Z":
        rotated = state
    elif basis == "X":
        rotated = apply_gate(HADAMARD, state)          # H rotates X-basis onto Z
    elif basis == "Y":
        rotated = apply_gate(HADAMARD, apply_gate(S_DAG, state))  # S-dagger then H
    return born_probabilities(rotated)

for basis in ("Z", "X"):
    p = probabilities_in_basis(KET_PLUS, basis)
    print(f"|+> measured in {basis}: P(0/+)={p['0']:.3f}  P(1/-)={p['1']:.3f}")

**Read the output.** In the $Z$ basis $|+\rangle$ is a fair coin ($0.5/0.5$) — the blindness from `01/03`. In the $X$ basis it is *certain*: $|+\rangle$ is the "$+$" outcome with probability $1$. Same state, two questions; the Hadamard chooses which one the measurement answers. Let's see it as a figure for a tilted state, where both bases are informative.

In [ ]:
psi = qubit_state(theta=np.pi / 3, phi=np.pi / 2)
bases = ["Z", "X", "Y"]
p0 = [probabilities_in_basis(psi, b)["0"] for b in bases]
p1 = [probabilities_in_basis(psi, b)["1"] for b in bases]

fig, ax = plt.subplots(figsize=(7.5, 4.2))
x = np.arange(len(bases)); w = 0.38
ax.bar(x - w / 2, p0, w, color=COLORS["source"], label="outcome 0 / +")
ax.bar(x + w / 2, p1, w, color=COLORS["target"], label="outcome 1 / -")
ax.set_xticks(x); ax.set_xticklabels([f"{b} basis" for b in bases])
ax.set_ylim(0, 1); ax.set_ylabel("probability")
ax.set_title("One state, three measurement bases", pad=12)
ax.legend()
plt.show()

**Read the figure.** The same state gives three different probability splits depending on the basis we rotate onto $Z$. No single measurement sees the whole state — each basis is one question. Stacking the $X$, $Y$, $Z$ answers is exactly how `01/15` reconstructs a full density matrix, and how `01/10` will read expectation values.

## 3. The Y basis needs an `sdg` first

The $Y$ basis is rotated onto $Z$ by applying $S^\dagger$ (the `sdg` gate, met properly in `01/09`) and *then* $H$. That two-step rotation is precisely the `qc.sdg(0); qc.h(0)` you will see in the tomography notebook `01/15` — no longer a mystery recipe, but a basis change you can derive.

In [ ]:
ket_plus_i = np.array([1, 1j], dtype=complex) / np.sqrt(2)   # the +1 eigenstate of Y
for basis in ("Z", "Y"):
    p = probabilities_in_basis(ket_plus_i, basis)
    print(f"|+i> measured in {basis}: P(0/+)={p['0']:.3f}  P(1/-)={p['1']:.3f}")

**Read the output.** The state $|+i\rangle$ is a fair coin in $Z$ but *certain* in the $Y$ basis — the `sdg`+`h` rotation lined its axis up with $Z$. Each Pauli observable from `01/07` has its own basis, and a small fixed rotation brings each one within reach of the only measurement hardware can do.

## Your turn

1. **Self-inverse.** Confirm $H^2 = I$ as a matrix, and explain in words why applying $H$ twice returns any state to itself.
2. **A classical state looks random in X.** Measure $|0\rangle$ in the $X$ basis with `probabilities_in_basis` and explain the $0.5/0.5$ result using $|0\rangle = \tfrac{1}{\sqrt2}(|+\rangle+|-\rangle)$.
3. **Build the X-basis circuit idea.** Write the gate sequence (in words or as matrices) that measures an arbitrary state in the $X$ basis, and check it matches `qc.h(0)` followed by a $Z$ measurement.

## What you built

- You applied the Hadamard and saw it map the $Z$ basis to the $X$ basis ($H|0\rangle=|+\rangle$, $HZH=X$), with $H^2=I$.
- You measured a qubit in the $X$ and $Y$ bases by rotating that basis onto $Z$ first — turning the one measurement hardware allows into any measurement you need.
- You explained the `qc.h(0)` that opens circuits and the `sdg`+`h` rotation `01/15` uses for $Y$.

The recipe that looked like magic in the tomography notebook is now something you can derive from scratch. That is real command of the toolkit.

## What's next

We used $S^\dagger$ in passing. In `01/09_phase_and_rotation_gates` we meet the phase gates $S, S^\dagger$ and the continuous rotations $R_x, R_y, R_z$ — the dials that send a qubit to *any* point on the Bloch sphere.

## References

- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, ch. 1.3.1, 2.2.5 (measurement in a basis), Cambridge University Press (2010).
- Qiskit textbook, "Single Qubit Gates" and "Measuring in a different basis".

**Previous:** `notebooks/01_QuantumFoundations/07_pauli_gates.ipynb` · **Next:** `notebooks/01_QuantumFoundations/09_phase_and_rotation_gates.ipynb`